In [ ]:
import os, cv2, random, shutil, glob, warnings, threading, gc, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization, LeakyReLU, Input, GlobalAveragePooling2D)
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.preprocessing.image import ImageDataGenerator

warnings.filterwarnings('ignore')

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'Memory growth enabled on {len(gpus)} GPU(s)')

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f'TensorFlow : {tf.__version__}')
print('Imports done')

In [ ]:
BASE_PATH = "./aptos_dataset"

APTOS_CSV = os.path.join(BASE_PATH, "train.csv")
APTOS_IMG_DIR = os.path.join(BASE_PATH, "train_images")

WORK_DIR = "./working"

OUT_PRE   = os.path.join(WORK_DIR, "preprocessed")
OUT_AUG   = os.path.join(WORK_DIR, "augmented")
FEAT_DIR  = os.path.join(WORK_DIR, "features")
MODEL_DIR = os.path.join(WORK_DIR, "models")
FL_LOG_DIR = os.path.join(WORK_DIR, "fl_logs")

for d in [WORK_DIR, OUT_PRE, OUT_AUG, FEAT_DIR, MODEL_DIR, FL_LOG_DIR]:
    os.makedirs(d, exist_ok=True)

CLASS_NAMES = {
    0: 'No_DR',
    1: 'Mild',
    2: 'Moderate',
    3: 'Severe',
    4: 'Proliferative_DR'
}

IMG_SIZE = (256, 256)
EFF_SIZE = (224, 224)
INPUT_SHAPE = (224, 224, 3)

TARGET_PER_CLASS = 6967
TRAIN_RATIO = 0.80

COLORS = [
    '#4CAF50',
    '#2196F3',
    '#FF9800',
    '#F44336',
    '#9C27B0'
]

FEAT_DIM = 1792
NUM_CLASSES = 5

BACKBONE_WARMUP_EPOCHS = 10
BACKBONE_FINETUNE_EPOCHS = 20

BACKBONE_LR_WARMUP = 1e-3
BACKBONE_LR_FINETUNE = 1e-4

BACKBONE_BATCH = 16
BACKBONE_UNFREEZE = 80

BATCH_SIZE = 128

DROPOUT = 0.4
L2_REG = 5e-5

LABEL_SMOOTHING = 0.1
ALPHA_MIXUP = 0.2

PRETRAIN_EPOCHS = 40
PRETRAIN_LR = 1e-3

NUM_CLIENTS = 5
FL_ROUNDS = 40
LOCAL_EPOCHS = 5
CLIENTS_PER_ROUND = 5

LR_FL_START = 5e-4
LR_FL_END = 1e-6

MU = 0.01
SERVER_MOMENTUM = 0.9
SERVER_FINETUNE_STEPS = 3

PATIENCE = 10

TTA_STEPS = 8


print("=" * 60)
print("Checking Dataset...")
print("=" * 60)

all_ok = True

paths = [
    ("APTOS CSV", APTOS_CSV),
    ("APTOS Images", APTOS_IMG_DIR)
]

for label, path in paths:
    ok = os.path.exists(path)
    print(f"{'OK' if ok else 'MISSING'}  {label}: {path}")

    if not ok:
        all_ok = False

if all_ok:

    image_count = len(glob.glob(os.path.join(APTOS_IMG_DIR, "*.png")))

    print(f"\nAPTOS Images Found : {image_count}")

    df = pd.read_csv(APTOS_CSV)

    print(f"CSV Records : {len(df)}")

    print("\nOriginal Class Distribution:")

    print(df["diagnosis"].value_counts().sort_index())

else:
    print("\nDataset verification failed. Please check the dataset paths.")

In [ ]:
aptos_df = pd.read_csv(APTOS_CSV)

aptos_df = aptos_df.rename(
    columns={
        'id_code': 'image_id',
        'diagnosis': 'label'
    }
)

aptos_df['image_path'] = aptos_df['image_id'].apply(
    lambda x: os.path.join(APTOS_IMG_DIR, f"{x}.png")
)

aptos_df['source'] = 'aptos2019'

aptos_df = aptos_df[['image_id', 'label', 'image_path', 'source']]

aptos_df['exists'] = aptos_df['image_path'].apply(os.path.exists)

missing = (~aptos_df['exists']).sum()

combined = (
    aptos_df[aptos_df['exists']]
    .drop(columns='exists')
    .reset_index(drop=True)
)


print(f'APTOS-2019 : {len(aptos_df):>6} rows')
print(f'Missing    : {missing}')
print(f'Valid      : {len(combined):>6} images\n')

print("Class Distribution")

for lbl, name in CLASS_NAMES.items():
    print(f'  Stage {lbl}  {name:20s}: {(combined["label"] == lbl).sum()}')

In [ ]:
def crop_black_border(img, tol=7):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    mask = gray > tol
    if mask.any():
        rows = np.where(mask.any(axis=1))[0]
        cols = np.where(mask.any(axis=0))[0]
        img  = img[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]
    return img

def gaussian_blur_subtraction(img, sigma=10):
    blur = cv2.GaussianBlur(img, (0, 0), sigma)
    return cv2.addWeighted(img, 4, blur, -4, 128)

def enhance_brightness_contrast(img, alpha=1.2, beta=10):
    return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

def preprocess(path):
    img = cv2.imread(path)
    if img is None: return None
    img = crop_black_border(img)
    img = gaussian_blur_subtraction(img)
    img = enhance_brightness_contrast(img)
    img = cv2.resize(img, IMG_SIZE)
    return img

def augment(img):
    img = img.copy()
    if random.random() > 0.5: img = cv2.flip(img, 1)
    if random.random() > 0.5: img = cv2.flip(img, 0)
    if random.random() > 0.3:
        angle = random.uniform(-30, 30)
        h, w  = img.shape[:2]
        M     = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        img   = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                               borderMode=cv2.BORDER_REFLECT)
    if random.random() > 0.4:
        h, w   = img.shape[:2]
        scale  = random.uniform(0.80, 1.00)
        nh, nw = int(h*scale), int(w*scale)
        top    = random.randint(0, h-nh)
        left   = random.randint(0, w-nw)
        img    = cv2.resize(img[top:top+nh, left:left+nw], (w, h))
    if random.random() > 0.5:
        img = cv2.convertScaleAbs(img, alpha=random.uniform(0.8, 1.2), beta=0)
    return img

print('Preprocessing & augmentation functions ready')

In [ ]:
for name in CLASS_NAMES.values():
    os.makedirs(os.path.join(OUT_PRE, name), exist_ok=True)

pre_records = []
lock = threading.Lock()

def process_one(row):
    img = preprocess(row['image_path'])

    if img is None:
        return None

    cls = CLASS_NAMES[row['label']]

    dst = os.path.join(
        OUT_PRE,
        cls,
        f"{row['source']}_{row['image_id']}.png"
    )

    cv2.imwrite(dst, img)

    return {
        'image_id': row['image_id'],
        'label': row['label'],
        'class_name': cls,
        'source': row['source'],
        'image_path': dst
    }

rows = [row for _, row in combined.iterrows()]

with ThreadPoolExecutor(max_workers=4) as executor:

    futures = [executor.submit(process_one, row) for row in rows]

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Preprocessing"
    ):
        result = future.result()

        if result is not None:
            with lock:
                pre_records.append(result)

pre_df = pd.DataFrame(pre_records)

manifest_path = os.path.join(OUT_PRE, "manifest.csv")
pre_df.to_csv(manifest_path, index=False)

print(f"\n{len(pre_df)} images preprocessed.")
print(f"Manifest saved to: {manifest_path}")

In [ ]:
aug_records = []

for lbl, name in CLASS_NAMES.items():
    src_paths = pre_df[pre_df['label'] == lbl]['image_path'].tolist()
    cls_dir   = OUT_AUG + '/' + name
    os.makedirs(cls_dir, exist_ok=True)
    count = 0

    for src in tqdm(src_paths, desc=f'Stage {lbl} {name:20s} (orig)', leave=False):
        dst = cls_dir + '/' + os.path.basename(src)
        shutil.copy2(src, dst)
        aug_records.append({'label':lbl, 'class_name':name, 'image_path':dst, 'augmented':False})
        count += 1

    idx  = 0
    pbar = tqdm(total=max(0, TARGET_PER_CLASS - count),
                desc=f'Stage {lbl} {name:20s} (aug)', leave=True)
    while count < TARGET_PER_CLASS:
        img = cv2.imread(random.choice(src_paths))
        if img is None: continue
        dst = cls_dir + '/aug_' + f'{idx:06d}_' + os.path.basename(random.choice(src_paths))
        cv2.imwrite(dst, augment(img))
        aug_records.append({'label':lbl, 'class_name':name, 'image_path':dst, 'augmented':True})
        count += 1; idx += 1; pbar.update(1)
    pbar.close()
    print(f'  Stage {lbl}  {name:20s}: {count}')

aug_df = pd.DataFrame(aug_records)
aug_df.to_csv(OUT_AUG + '/manifest.csv', index=False)
print(f'\nTotal: {len(aug_df)} images')
for lbl, name in CLASS_NAMES.items():
    print(f'  Stage {lbl}  {name:20s}: {(aug_df["label"]==lbl).sum()}')

print('\nDeleting preprocessed temp folder...')
shutil.rmtree(OUT_PRE)

In [ ]:
all_aug_labels = aug_df['label'].values
aug_indices = np.arange(len(aug_df))

bb_train_idx, bb_val_idx = train_test_split(
    aug_indices,
    test_size=0.20,
    stratify=all_aug_labels,
    random_state=SEED
)

bb_train_df = aug_df.iloc[bb_train_idx].reset_index(drop=True)
bb_val_df = aug_df.iloc[bb_val_idx].reset_index(drop=True)

BB_TRAIN_DIR = os.path.join(WORK_DIR, "bb_train")
BB_VAL_DIR = os.path.join(WORK_DIR, "bb_val")

for d in [BB_TRAIN_DIR, BB_VAL_DIR]:
    for name in CLASS_NAMES.values():
        os.makedirs(os.path.join(d, name), exist_ok=True)

for _, row in tqdm(bb_train_df.iterrows(),
                   total=len(bb_train_df),
                   desc="Linking train"):

    dst = os.path.join(
        BB_TRAIN_DIR,
        row["class_name"],
        os.path.basename(row["image_path"])
    )

    if not os.path.exists(dst):
        shutil.copy2(row["image_path"], dst)

for _, row in tqdm(bb_val_df.iterrows(),
                   total=len(bb_val_df),
                   desc="Linking val"):

    dst = os.path.join(
        BB_VAL_DIR,
        row["class_name"],
        os.path.basename(row["image_path"])
    )

    if not os.path.exists(dst):
        shutil.copy2(row["image_path"], dst)

print(f"Backbone train: {len(bb_train_df)} | Val: {len(bb_val_df)}")

In [ ]:
def build_efficientnet_model(trainable_base=False, lr=BACKBONE_LR_WARMUP):
    base = EfficientNetB4(weights='imagenet', include_top=False,
                           input_shape=INPUT_SHAPE, pooling='avg')
    for layer in base.layers:
        layer.trainable = False

    if trainable_base:
        for layer in base.layers[-BACKBONE_UNFREEZE:]:
            if not isinstance(layer, BatchNormalization):
                layer.trainable = True

    x      = base.output
    x      = BatchNormalization()(x)
    x      = Dense(1024, kernel_regularizer=l2(L2_REG))(x)
    x      = LeakyReLU(negative_slope=0.1)(x)
    x      = BatchNormalization()(x)
    x      = Dropout(DROPOUT)(x)
    x      = Dense(512, kernel_regularizer=l2(L2_REG))(x)
    x      = LeakyReLU(negative_slope=0.1)(x)
    x      = BatchNormalization()(x)
    x      = Dropout(0.3)(x)
    output = Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    m = Model(inputs=base.input, outputs=output)
    m.compile(
        optimizer = Adam(learning_rate=lr),
        loss      = CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics   = ['accuracy']
    )
    return m, base


def make_generators(train_dir, val_dir, batch_size):
    dg_train = ImageDataGenerator(preprocessing_function=eff_preprocess)
    dg_val   = ImageDataGenerator(preprocessing_function=eff_preprocess)
    gen_train = dg_train.flow_from_directory(
        train_dir, target_size=EFF_SIZE, batch_size=batch_size,
        class_mode='categorical', shuffle=True, seed=SEED)
    gen_val = dg_val.flow_from_directory(
        val_dir, target_size=EFF_SIZE, batch_size=batch_size,
        class_mode='categorical', shuffle=False)
    return gen_train, gen_val


train_labels_bb = np.load(FEAT_DIR + '/train_pool/labels.npy') \
    if os.path.exists(FEAT_DIR + '/train_pool/labels.npy') else bb_train_df['label'].values
cw_arr_bb = compute_class_weight('balanced',
                                  classes=np.unique(bb_train_df['label'].values),
                                  y=bb_train_df['label'].values)
cw_dict_bb = {i: float(w) for i, w in enumerate(cw_arr_bb)}

print('Class weights for backbone training:')
for k, v in cw_dict_bb.items(): print(f'  {CLASS_NAMES[k]:20s}: {v:.4f}')

In [ ]:
print('Phase 1: Backbone frozen — training head only...')
full_model, base_model = build_efficientnet_model(
    trainable_base=False, lr=BACKBONE_LR_WARMUP)

trainable = sum(p.numpy().size for p in full_model.trainable_weights)
total     = sum(p.numpy().size for p in full_model.weights)
print(f'  Trainable: {trainable:,}  /  Total: {total:,}')

gen_train, gen_val = make_generators(BB_TRAIN_DIR, BB_VAL_DIR, BACKBONE_BATCH)

cb_p1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_DIR + '/eff_phase1.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.CSVLogger(FL_LOG_DIR + '/backbone_phase1.csv')
]

h_p1 = full_model.fit(
    gen_train,
    epochs          = BACKBONE_WARMUP_EPOCHS,
    validation_data = gen_val,
    class_weight    = cw_dict_bb,
    callbacks       = cb_p1,
    verbose         = 1
)
print(f'Phase 1 best val acc: {max(h_p1.history["val_accuracy"])*100:.2f}%')

In [ ]:
print(f'\nPhase 2: Unfreezing last {BACKBONE_UNFREEZE} backbone layers...')
for layer in base_model.layers[-BACKBONE_UNFREEZE:]:
    if not isinstance(layer, BatchNormalization):
        layer.trainable = True

full_model.compile(
    optimizer = Adam(learning_rate=BACKBONE_LR_FINETUNE),
    loss      = CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics   = ['accuracy']
)

trainable_ft = sum(p.numpy().size for p in full_model.trainable_weights)
print(f'  Trainable now: {trainable_ft:,}')

gen_train, gen_val = make_generators(BB_TRAIN_DIR, BB_VAL_DIR, BACKBONE_BATCH)

cb_p2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_DIR + '/eff_finetuned.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=4, min_lr=1e-8, verbose=1),
    tf.keras.callbacks.CSVLogger(FL_LOG_DIR + '/backbone_phase2.csv')
]

h_p2 = full_model.fit(
    gen_train,
    epochs          = BACKBONE_FINETUNE_EPOCHS,
    validation_data = gen_val,
    class_weight    = cw_dict_bb,
    callbacks       = cb_p2,
    verbose         = 1
)

backbone_best_acc = max(h_p2.history['val_accuracy'])
print(f'\nBackbone fine-tuned best val acc: {backbone_best_acc*100:.2f}%')
full_model.save(MODEL_DIR + '/eff_finetuned_final.keras')

In [ ]:
acc      = h_p1.history['accuracy']     + h_p2.history['accuracy']
val_acc  = h_p1.history['val_accuracy'] + h_p2.history['val_accuracy']
ep_range = range(1, len(acc)+1)
p1_end   = len(h_p1.history['accuracy'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ep_range, [v*100 for v in acc],     label='Train Acc', linewidth=2)
ax.plot(ep_range, [v*100 for v in val_acc], label='Val Acc',   linewidth=2)
ax.axvline(p1_end, color='gray', linestyle='--', alpha=0.7, label='Fine-tune starts')
ax.axhline(90, color='#4CAF50', linestyle=':', linewidth=1.5, label='90% target')
ax.set_title('EfficientNetB4 Backbone Fine-Tuning', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FL_LOG_DIR + '/backbone_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nBackbone fine-tuning already achieves: {backbone_best_acc*100:.2f}%')
print('Now extracting features from this fine-tuned backbone for FL...')

In [ ]:
fine_tuned_extractor = Model(
    inputs=base_model.input,
    outputs=base_model.output
)

fine_tuned_extractor.trainable = False

FEAT_DIM = fine_tuned_extractor.output_shape[-1]

print(f"Fine-tuned feature vector: {FEAT_DIM} dimensions")

print("Deleting backbone split folders...")

if os.path.exists(BB_TRAIN_DIR):
    shutil.rmtree(BB_TRAIN_DIR)

if os.path.exists(BB_VAL_DIR):
    shutil.rmtree(BB_VAL_DIR)

all_features = []
all_labels = []

EXTRACT_BATCH = 128

for lbl, name in CLASS_NAMES.items():

    class_dir = os.path.join(OUT_AUG, name)

    img_paths = (
        sorted(glob.glob(os.path.join(class_dir, "*.png"))) +
        sorted(glob.glob(os.path.join(class_dir, "*.jpg"))) +
        sorted(glob.glob(os.path.join(class_dir, "*.jpeg")))
    )

    n = len(img_paths)

    print(f"Extracting Stage {lbl} ({name}): {n} images...")

    class_features = []

    for start in tqdm(
        range(0, n, EXTRACT_BATCH),
        desc=f"Stage {lbl}",
        leave=False
    ):

        batch_paths = img_paths[start:start + EXTRACT_BATCH]

        batch_imgs = []

        for p in batch_paths:

            img = cv2.imread(p)

            if img is None:
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, EFF_SIZE)

            batch_imgs.append(img)

        # Skip empty batches
        if len(batch_imgs) == 0:
            continue

        batch_arr = eff_preprocess(
            np.asarray(batch_imgs, dtype=np.float32)
        )

        feats = fine_tuned_extractor.predict(
            batch_arr,
            verbose=0
        )

        class_features.append(feats)

    if len(class_features) == 0:
        print(f"Warning: No valid images found for {name}")
        continue

    class_features = np.concatenate(class_features, axis=0)

    all_features.append(class_features)

    all_labels.append(
        np.full(len(class_features), lbl, dtype=np.int32)
    )

    print(f"    -> {class_features.shape}")


all_features = np.concatenate(all_features, axis=0).astype(np.float32)
all_labels = np.concatenate(all_labels, axis=0).astype(np.int32)

print(f"\nAll features: {all_features.shape}")

np.save(
    os.path.join(FEAT_DIR, "all_features.npy"),
    all_features
)

np.save(
    os.path.join(FEAT_DIR, "all_labels.npy"),
    all_labels
)

del fine_tuned_extractor
del full_model

gc.collect()
tf.keras.backend.clear_session()

print("\nDeleting augmented images (features saved)...")

if os.path.exists(OUT_AUG):
    shutil.rmtree(OUT_AUG)

print("Fine-tuned features extracted.")
print("Backbone freed from GPU.")

In [ ]:
print(os.path.exists(os.path.join(FEAT_DIR, "all_features.npy")))
print(os.path.exists(os.path.join(FEAT_DIR, "all_labels.npy")))

all_features = np.load(os.path.join(FEAT_DIR, "all_features.npy"))
all_labels = np.load(os.path.join(FEAT_DIR, "all_labels.npy"))

print(all_features.shape)
print(all_labels.shape)

In [ ]:
indices = np.arange(len(all_labels))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    stratify=all_labels,
    random_state=SEED
)

X_train_all = all_features[train_idx]
y_train_all = all_labels[train_idx]

X_test = all_features[test_idx]
y_test = all_labels[test_idx]

# One-hot encoding
y_train_oh = tf.keras.utils.to_categorical(
    y_train_all,
    NUM_CLASSES
)

y_test_oh = tf.keras.utils.to_categorical(
    y_test,
    NUM_CLASSES
)


cw_arr = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_all),
    y=y_train_all
)

cw_dict = {
    i: float(w)
    for i, w in enumerate(cw_arr)
}


print(f"Train: {X_train_all.shape}")
print(f"Test : {X_test.shape}\n")

print("Train Class Distribution:")

for lbl, name in CLASS_NAMES.items():
    print(
        f"  Stage {lbl}  {name:20s}: "
        f"{(y_train_all == lbl).sum()}"
    )

client_X = []
client_y = []

remaining_X = X_train_all.copy()
remaining_y = y_train_all.copy()

for c in range(NUM_CLIENTS):

    if c == NUM_CLIENTS - 1:

        cX = remaining_X
        cy = remaining_y

    else:

        frac = 1.0 / (NUM_CLIENTS - c)

        idx = np.arange(len(remaining_y))

        client_idx, remain_idx = train_test_split(
            idx,
            test_size=1 - frac,
            stratify=remaining_y,
            random_state=SEED + c
        )

        cX = remaining_X[client_idx]
        cy = remaining_y[client_idx]

        remaining_X = remaining_X[remain_idx]
        remaining_y = remaining_y[remain_idx]

    client_X.append(cX)
    client_y.append(cy)

    print(f"Client {c}: {len(cy)} samples")

for c in range(NUM_CLIENTS):

    client_dir = os.path.join(
        FEAT_DIR,
        f"client_{c}"
    )

    os.makedirs(client_dir, exist_ok=True)

    np.save(
        os.path.join(client_dir, "features.npy"),
        client_X[c]
    )

    np.save(
        os.path.join(client_dir, "labels.npy"),
        client_y[c]
    )

train_pool_dir = os.path.join(
    FEAT_DIR,
    "train_pool"
)

os.makedirs(train_pool_dir, exist_ok=True)

np.save(
    os.path.join(train_pool_dir, "features.npy"),
    X_train_all
)

np.save(
    os.path.join(train_pool_dir, "labels.npy"),
    y_train_all
)

del all_features
del all_labels

gc.collect()

In [ ]:
def mixup_batch(X, y_oh, alpha=ALPHA_MIXUP):
    if alpha <= 0: return X, y_oh
    lam   = np.random.beta(alpha, alpha)
    idx   = np.random.permutation(len(X))
    X_mix = lam * X + (1 - lam) * X[idx]
    y_mix = lam * y_oh + (1 - lam) * y_oh[idx]
    return X_mix.astype(np.float32), y_mix.astype(np.float32)


def build_head_model(lr=PRETRAIN_LR, use_adam=True):
    inputs = Input(shape=(FEAT_DIM,), name='features')
    x = BatchNormalization()(inputs)
    x = Dense(1024, kernel_regularizer=l2(L2_REG))(x)
    x = LeakyReLU(negative_slope=0.1)(x)
    x = BatchNormalization()(x)
    x = Dropout(DROPOUT)(x)
    x = Dense(512, kernel_regularizer=l2(L2_REG))(x)
    x = LeakyReLU(negative_slope=0.1)(x)
    x = BatchNormalization()(x)
    x = Dropout(DROPOUT)(x)
    x = Dense(256, kernel_regularizer=l2(L2_REG))(x)
    x = LeakyReLU(negative_slope=0.1)(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    output = Dense(NUM_CLASSES, activation='softmax')(x)

    m   = Model(inputs=inputs, outputs=output)
    opt = Adam(learning_rate=lr) if use_adam else \
          SGD(learning_rate=lr, momentum=0.9, nesterov=True)
    m.compile(
        optimizer = opt,
        loss      = CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics   = ['accuracy']
    )
    return m


tmp = build_head_model()
print(f'Head params: {tmp.count_params():,}')
del tmp; gc.collect()

In [ ]:
print(f'Central pre-training on {len(y_train_all)} samples with Mixup...')
pretrained_head = build_head_model(lr=PRETRAIN_LR, use_adam=True)

best_pretrain_acc = 0.0
best_pretrain_w   = None
no_imp_pretrain   = 0
pretrain_history  = {'epoch':[], 'loss':[], 'acc':[], 'val_acc':[]}

optimizer = Adam(learning_rate=PRETRAIN_LR)
loss_fn   = CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)

for epoch in range(PRETRAIN_EPOCHS):
    perm      = np.random.permutation(len(y_train_all))
    X_shuf    = X_train_all[perm]
    y_shuf    = y_train_oh[perm]

    epoch_losses = []
    for start in range(0, len(X_shuf), BATCH_SIZE):
        Xb    = X_shuf[start : start + BATCH_SIZE]
        yb    = y_shuf[start : start + BATCH_SIZE]
        Xb, yb = mixup_batch(Xb, yb)
        sw    = tf.constant([cw_dict[int(np.argmax(l))] for l in yb], dtype=tf.float32)

        with tf.GradientTape() as tape:
            preds = pretrained_head(Xb, training=True)
            loss  = loss_fn(yb, preds, sample_weight=sw)
        grads = tape.gradient(loss, pretrained_head.trainable_weights)
        optimizer.apply_gradients(zip(grads, pretrained_head.trainable_weights))
        epoch_losses.append(float(loss))

    val_preds   = pretrained_head(X_test, training=False).numpy()
    val_acc     = float(np.mean(np.argmax(val_preds, axis=1) == y_test))
    epoch_loss  = float(np.mean(epoch_losses))

    pretrain_history['epoch'].append(epoch+1)
    pretrain_history['loss'].append(epoch_loss)
    pretrain_history['acc'].append(val_acc)
    pretrain_history['val_acc'].append(val_acc)

    if (epoch+1) % 5 == 0 or val_acc > best_pretrain_acc:
        print(f'  Epoch {epoch+1:3d}/{PRETRAIN_EPOCHS} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc*100:.2f}%')

    if val_acc > best_pretrain_acc:
        best_pretrain_acc = val_acc
        best_pretrain_w   = [w.copy() for w in pretrained_head.get_weights()]
        pretrained_head.save(MODEL_DIR + '/pretrained_head.keras')
        no_imp_pretrain = 0
    else:
        no_imp_pretrain += 1

    # Reduce LR
    if no_imp_pretrain == 5:
        new_lr = float(optimizer.learning_rate) * 0.3
        optimizer.learning_rate.assign(max(new_lr, 1e-7))
        print(f'  LR reduced to {float(optimizer.learning_rate):.2e}')

    if no_imp_pretrain >= 10:
        print(f'  Early stopping at epoch {epoch+1}')
        break

pretrained_head.set_weights(best_pretrain_w)
pretrain_val_acc = best_pretrain_acc
PRETRAINED_WEIGHTS = [w.copy() for w in pretrained_head.get_weights()]
del pretrained_head; gc.collect(); tf.keras.backend.clear_session()
print(f'\nPre-train best val accuracy: {pretrain_val_acc*100:.2f}%')

In [ ]:
def cosine_lr(fl_round, total_rounds, lr_start, lr_end):
    return lr_end + 0.5 * (lr_start - lr_end) * \
           (1 + math.cos(math.pi * fl_round / total_rounds))


def load_client_data(client_id):
    X = np.load(FEAT_DIR + f'/client_{client_id}/features.npy').astype(np.float32)
    y = np.load(FEAT_DIR + f'/client_{client_id}/labels.npy').astype(np.int32)
    return X, y


def fedprox_local_train(global_weights, client_id, local_epochs, lr, mu=MU):
    X, y      = load_client_data(client_id)
    n_samples = len(y)
    y_oh      = tf.keras.utils.to_categorical(y, NUM_CLASSES)

    cw_arr_c  = compute_class_weight('balanced', classes=np.unique(y), y=y)
    cw_dict_c = {i: float(w) for i, w in enumerate(cw_arr_c)}

    local_model = build_head_model(lr=lr, use_adam=False)
    local_model.set_weights(global_weights)

    # Fixed: find trainable weight indices exactly
    all_w_ids      = [id(w) for w in local_model.weights]
    trainable_ids  = set(id(w) for w in local_model.trainable_weights)
    trainable_idxs = [i for i, wid in enumerate(all_w_ids) if wid in trainable_ids]
    global_trainable_tensors = [
        tf.constant(global_weights[i], dtype=tf.float32)
        for i in trainable_idxs
    ]

    optimizer = SGD(learning_rate=lr, momentum=0.9, nesterov=True)
    loss_fn   = CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)

    for epoch in range(local_epochs):
        perm      = np.random.permutation(len(y))
        X_shuf    = X[perm]
        y_shuf    = y_oh[perm]

        for start in range(0, len(X_shuf), BATCH_SIZE):
            Xb     = X_shuf[start : start + BATCH_SIZE]
            yb     = y_shuf[start : start + BATCH_SIZE]
            Xb, yb = mixup_batch(Xb, yb)   # Mixup on client data
            sw     = tf.constant(
                [cw_dict_c.get(int(np.argmax(l)), 1.0) for l in yb],
                dtype=tf.float32)

            with tf.GradientTape() as tape:
                preds     = local_model(Xb, training=True)
                task_loss = loss_fn(yb, preds, sample_weight=sw)
                prox      = tf.constant(0.0)
                for w_loc, w_glob in zip(local_model.trainable_weights,
                                          global_trainable_tensors):
                    prox = prox + tf.reduce_sum(tf.square(w_loc - w_glob))
                total_loss = task_loss + (mu / 2.0) * prox

            grads = tape.gradient(total_loss, local_model.trainable_weights)
            optimizer.apply_gradients(zip(grads, local_model.trainable_weights))

    updated_weights = [w.numpy().copy() for w in local_model.weights]
    del local_model, X, y, y_oh, global_trainable_tensors
    gc.collect(); tf.keras.backend.clear_session()
    return updated_weights, n_samples


def fed_avg(global_weights, client_weights_list, client_sizes):
    total = sum(client_sizes)
    new_w = []
    for i in range(len(global_weights)):
        ws = np.zeros_like(global_weights[i], dtype=np.float64)
        for cw, n in zip(client_weights_list, client_sizes):
            ws += cw[i].astype(np.float64) * (n / total)
        new_w.append(ws.astype(np.float32))
    return new_w


def fedavgm_update(global_weights, aggregated_weights, velocity, beta=SERVER_MOMENTUM):
    new_velocity = []
    new_weights  = []
    for i in range(len(global_weights)):
        delta = aggregated_weights[i] - global_weights[i]
        v_new = beta * velocity[i] + delta
        new_velocity.append(v_new)
        new_weights.append(global_weights[i] + v_new)
    return new_weights, new_velocity


print('All FL functions ready')

In [ ]:
global_head  = build_head_model(lr=LR_FL_START, use_adam=False)
global_head.set_weights(PRETRAINED_WEIGHTS)
velocity     = [np.zeros_like(w) for w in PRETRAINED_WEIGHTS]

fl_history   = {'round':[], 'global_acc':[], 'global_loss':[], 'lr':[]}
best_acc     = 0.0
best_weights = None
no_improve   = 0

print('=' * 70)
print('  FEDERATED LEARNING — Fine-tuned EfficientNetB4 + FedProx + Mixup')
print(f'  Clients:{NUM_CLIENTS}  Rounds:{FL_ROUNDS}  LocalEpochs:{LOCAL_EPOCHS}')
print(f'  LR: {LR_FL_START}→{LR_FL_END} (cosine)  mu:{MU}  beta:{SERVER_MOMENTUM}')
print(f'  Backbone val acc    : {backbone_best_acc*100:.2f}%')
print(f'  Pre-train val acc   : {pretrain_val_acc*100:.2f}%')
print(f'  Train samples       : {len(y_train_all)}  |  Test: {len(y_test)}')
print('=' * 70)

for fl_round in range(1, FL_ROUNDS + 1):

    current_lr     = cosine_lr(fl_round, FL_ROUNDS, LR_FL_START, LR_FL_END)
    global_weights = [w.numpy().copy() if hasattr(w, 'numpy') else w.copy()
                      for w in global_head.weights]

    client_weights_list = []
    round_client_sizes  = []
    selected_clients    = random.sample(range(NUM_CLIENTS), CLIENTS_PER_ROUND)

    pbar = tqdm(selected_clients,
                desc=f'Round {fl_round:02d}/{FL_ROUNDS} lr={current_lr:.1e}',
                leave=True)

    for c in pbar:
        w, n = fedprox_local_train(
            global_weights, client_id=c,
            local_epochs=LOCAL_EPOCHS, lr=current_lr, mu=MU)
        client_weights_list.append(w)
        round_client_sizes.append(n)
        pbar.set_postfix({'c': c, 'n': n})

    aggregated_weights            = fed_avg(global_weights, client_weights_list, round_client_sizes)
    new_global_weights, velocity  = fedavgm_update(
        global_weights, aggregated_weights, velocity, beta=SERVER_MOMENTUM)

    del global_head; gc.collect(); tf.keras.backend.clear_session()
    global_head = build_head_model(lr=current_lr * 0.1, use_adam=False)
    global_head.set_weights(new_global_weights)

    # Server-side fine-tune with Mixup
    if SERVER_FINETUNE_STEPS > 0:
        perm   = np.random.permutation(len(y_train_all))
        Xs, ys = mixup_batch(X_train_all[perm], y_train_oh[perm])
        global_head.fit(Xs, ys, epochs=SERVER_FINETUNE_STEPS,
                        batch_size=BATCH_SIZE, class_weight=cw_dict, verbose=0)

    g_loss, g_acc = global_head.evaluate(X_test, y_test_oh, batch_size=256, verbose=0)

    fl_history['round'].append(fl_round)
    fl_history['global_acc'].append(g_acc)
    fl_history['global_loss'].append(g_loss)
    fl_history['lr'].append(current_lr)

    print(f'  Round {fl_round:02d} | Acc: {g_acc*100:.2f}% | '
          f'Loss: {g_loss:.4f} | LR: {current_lr:.1e}')

    if g_acc > best_acc:
        best_acc     = g_acc
        best_weights = [w.numpy().copy() if hasattr(w, 'numpy') else w.copy()
                        for w in global_head.weights]
        global_head.save(MODEL_DIR + '/fl_best.keras')
        print(f'  New best! Acc: {best_acc*100:.2f}%')
        no_improve = 0
    else:
        no_improve += 1

    pd.DataFrame(fl_history).to_csv(FL_LOG_DIR + '/fl_log.csv', index=False)

    if no_improve >= PATIENCE:
        print(f'\nEarly stopping at round {fl_round}')
        break

print(f'\nFL complete!  Best accuracy: {best_acc*100:.2f}%')

In [ ]:
global_head.set_weights(best_weights)

final_loss, final_acc = global_head.evaluate(X_test, y_test_oh, batch_size=256, verbose=0)
print(f'Standard Accuracy      : {final_acc*100:.2f}%')

# TTA evaluation — apply small feature-level noise and average predictions
def tta_predict(model, X, n_steps=TTA_STEPS, noise_std=0.02):

    all_preds = []
    all_preds.append(model(X, training=False).numpy())

    for _ in range(n_steps - 1):
        noise = np.random.normal(0, noise_std, X.shape).astype(np.float32)
        all_preds.append(model(X + noise, training=False).numpy())
    return np.mean(all_preds, axis=0)

print(f'\nRunning TTA ({TTA_STEPS} steps)...')
tta_probs = tta_predict(global_head, X_test, n_steps=TTA_STEPS)
tta_preds = np.argmax(tta_probs, axis=1)
tta_acc   = float(np.mean(tta_preds == y_test))

print(f'TTA Accuracy ({TTA_STEPS} steps): {tta_acc*100:.2f}%')
print(f'TTA gain over standard    : +{(tta_acc - final_acc)*100:.2f}%')

y_pred = tta_preds

print('\n' + '='*70)
print('  CLASSIFICATION REPORT (with TTA)')
print('='*70)
print(classification_report(y_test, y_pred, target_names=list(CLASS_NAMES.values())))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES.values(), yticklabels=CLASS_NAMES.values(),
            linewidths=0.5, ax=ax)
ax.set_title('FL Global Model — Confusion Matrix (TTA)',
             fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.xticks(rotation=20, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(FL_LOG_DIR + '/confusion_matrix_tta.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
log_df = pd.DataFrame(fl_history)
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].plot(log_df['round'], log_df['global_acc']*100,
             color='#2196F3', linewidth=2, marker='o', markersize=4, label='FL Acc')
axes[0].axhline(backbone_best_acc*100, color='#9C27B0', linestyle='--',
                linewidth=1.5, label=f'Backbone {backbone_best_acc*100:.1f}%')
axes[0].axhline(pretrain_val_acc*100, color='#FF9800', linestyle='--',
                linewidth=1.5, label=f'Pre-train {pretrain_val_acc*100:.1f}%')
axes[0].axhline(90, color='#4CAF50', linestyle=':', linewidth=1.5, label='90% target')
axes[0].axhline(tta_acc*100, color='#00BCD4', linestyle=':',
                linewidth=1.5, label=f'FL+TTA {tta_acc*100:.1f}%')
axes[0].set_title('FL Global Accuracy per Round', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Round'); axes[0].set_ylabel('Accuracy (%)')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(log_df['round'], log_df['global_loss'],
             color='#F44336', linewidth=2, marker='o', markersize=4)
axes[1].set_title('FL Global Loss per Round', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Round'); axes[1].set_ylabel('Loss'); axes[1].grid(alpha=0.3)

axes[2].plot(log_df['round'], log_df['lr'],
             color='#FF9800', linewidth=2, marker='o', markersize=4)
axes[2].set_title('Cosine LR Schedule', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Round'); axes[2].set_ylabel('LR')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.suptitle('Federated Learning — EfficientNetB4 + FedProx + Mixup + TTA',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FL_LOG_DIR + '/fl_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
global_head.save(os.path.join(MODEL_DIR, "fl_final.keras"))

best_round = (
    fl_history["global_acc"].index(max(fl_history["global_acc"])) + 1
)

print("\n" + "=" * 70)
print("  FINAL SUMMARY — FL + EfficientNetB4 + Mixup + TTA")
print("=" * 70)

print(f"  Dataset        : {len(combined)} images → {len(aug_df)} after balance")
print(f"  Backbone       : EfficientNetB4 (fine-tuned {BACKBONE_UNFREEZE} layers)")
print(f"  Features       : {FEAT_DIM}-dim domain-adapted vectors")
print(f"  FL Algorithm   : FedProx (mu={MU}) + FedAvgM (beta={SERVER_MOMENTUM})")
print(f"  Training       : Mixup (alpha={ALPHA_MIXUP})")
print(f"  Inference      : TTA ({TTA_STEPS} steps)")

print("=" * 70)

print(f"  Backbone val acc    : {backbone_best_acc * 100:.2f}%")
print(f"  Pre-train val acc   : {pretrain_val_acc * 100:.2f}%")
print(f"  FL best round       : {best_round}")
print(f"  FL best acc         : {best_acc * 100:.2f}%")
print(f"  FL + TTA acc        : {tta_acc * 100:.2f}%")

print("=" * 70)

print("  Central v1 baseline : 67.20%")
print("  Paper target        : 89.21%")
print(
    f"  Our result (FL+TTA) : {tta_acc * 100:.2f}% "
    f"{' >90%' if tta_acc > 0.90 else ''}"
)

print("=" * 70)